In [ ]:
# memory_management.py
import json
import os
from datetime import datetime
from typing import Dict, Any, List

MEMORY_FILE = "rua_multi_user_memory.json"

class RUACognitiveHub:
    def __init__(self, storage_path: str = MEMORY_FILE):
        self.storage_path = storage_path
        self.active_user = "Guest"
        self.memory = self._load_from_disk()
        
        # TIER 1: Working Memory (Session specific)
        self.working_memory = {"buffer": {}, "last_search": None}

    def _load_from_disk(self) -> Dict[str, Any]:
        if os.path.exists(self.storage_path):
            with open(self.storage_path, "r") as f:
                return json.load(f)
        return {"users": {}}

    def _ensure_user_exists(self, user_id: str):
        if user_id not in self.memory["users"]:
            self.memory["users"][user_id] = {
                "critical_info": {},
                "semantic_memory": {},
                "episodic_memory": [],
                "created_at": datetime.now().strftime("%Y-%m-%d")
            }
            self.save_to_disk()

    def set_active_user(self, user_id: str):
        self.active_user = user_id
        self._ensure_user_exists(user_id)
        print(f"👤 [IDENTITY] Switched to profile: {user_id}")

    def save_to_disk(self):
        with open(self.storage_path, "w") as f:
            json.dump(self.memory, f, indent=4)

    def learn_fact(self, key: str, value: Any, tier: str = "semantic"):
        user_data = self.memory["users"][self.active_user]
        if tier == "critical":
            user_data["critical_info"][key] = value
        else:
            user_data["semantic_memory"][key] = value
        self.save_to_disk()

    def get_brain_dump(self) -> str:
        """Returns context strictly for the active user."""
        if self.active_user == "Guest": return "User is unknown."
        
        u = self.memory["users"][self.active_user]
        dump = f"Current User: {self.active_user}. "
        if u['critical_info']: dump += f"CRITICAL: {u['critical_info']}. "
        if u['semantic_memory']: dump += f"FACTS: {u['semantic_memory']}. "
        if self.working_memory['buffer']: dump += f"ACTIVE TASK: {self.working_memory['buffer']}. "
        return dump

    def consolidate(self, llm):
        """Distills episodic memory for the active user only."""
        user_data = self.memory["users"][self.active_user]
        episodes = user_data["episodic_memory"]
        if len(episodes) < 1: return "Archives lean. No consolidation needed."
        
        # Distillation logic (same as previous version, but scoped to user)
        prompt = f"Distill these memories for {self.active_user}: {episodes}. Return JSON facts."
        try:
            res = llm.invoke(prompt)
            facts = json.loads(res.content.replace("```json", "").replace("```", "").strip())
            user_data["semantic_memory"].update(facts)
            user_data["episodic_memory"] = user_data["episodic_memory"][-1:]
            self.save_to_disk()
            return f"Brain optimized for {self.active_user}."
        except: return "Consolidation skipped."

rua_brain = RUACognitiveHub()

def identity_router(user_input: str):
    """Detects name changes or login triggers."""
    text = user_input.lower()
    if "i am" in text or "my name is" in text:
        name = user_input.split("is")[-1].strip() if "is" in text else user_input.split("am")[-1].strip()
        rua_brain.set_active_user(name.capitalize())
        return f"Identity confirmed: {rua_brain.active_user}"
    return None

In [ ]:
GOOGLE_API_KEY=''

In [ ]:
# memory diagnostics logic 

import os
from langchain_google_genai import ChatGoogleGenerativeAI
# Assuming memory_management is in the same notebook or imported
# from memory_management import rua_brain, identity_router
api_key = GOOGLE_API_KEY

def run_rua_diagnostics():
    print("🧪 --- INITIATING RUA CORE DIAGNOSTICS --- 🧪\n")

    # TEST 1: Identity & Storage Creation
    print("▶️ TEST 1: Identity Router")
    router_response = identity_router("Hello, my name is Tester")
    print(f"Router Output: {router_response}")
    print(f"Active User in Brain: {rua_brain.active_user}")
    assert rua_brain.active_user == "Tester", "❌ Identity routing failed."
    print("✅ Identity Check Passed.\n")

    # TEST 2: Semantic Memory Storage
    print("▶️ TEST 2: Learning Facts")
    rua_brain.learn_fact("test_status", "All systems operational", tier="semantic")
    dump = rua_brain.get_brain_dump()
    print(f"Brain Dump: {dump}")
    assert "operational" in dump, "❌ Memory storage or retrieval failed."
    print("✅ Memory Storage Passed.\n")

    # TEST 3: Episodic Memory & Consolidation
    print("▶️ TEST 3: Cloud Consolidation")
    try:
        # Fake a conversation history
        rua_brain.memory["users"]["Tester"]["episodic_memory"].extend([
            "User Tester logged in.",
            "User Tester said they are building an AI.",
            "User Tester asked to test the system."
        ])
        
        # Fire up the cloud LLM
        cloud_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=GOOGLE_API_KEY)
        consolidation_result = rua_brain.consolidate(cloud_llm)
        
        print(f"Consolidation Output: {consolidation_result}")
        print(f"Updated Semantic Memory: {rua_brain.memory['users']['Tester']['semantic_memory']}")
        print("✅ Consolidation Passed.\n")
    except Exception as e:
        print(f"❌ Consolidation Failed: {e}")

    print("🎉 DIAGNOSTICS COMPLETE. Check 'rua_multi_user_memory.json' in your folder.")

run_rua_diagnostics()

✅ API Key found! It starts with: AIzaSyBXT3...
🧪 --- INITIATING RUA CORE DIAGNOSTICS --- 🧪

▶️ TEST 1: Identity Router
👤 [IDENTITY] Switched to profile: Tester
Router Output: Identity confirmed: Tester
Active User in Brain: Tester
✅ Identity Check Passed.

▶️ TEST 2: Learning Facts
Brain Dump: Current User: Tester. FACTS: {'test_status': 'All systems operational', 'facts_about_Tester': ['Tester logged in.', 'Tester is building an AI.', 'Tester asked to test the system.'], 'entity': 'fact'}. 
✅ Memory Storage Passed.

▶️ TEST 3: Cloud Consolidation
Consolidation Output: Brain optimized for Tester.
Updated Semantic Memory: {'test_status': 'All systems operational', 'facts_about_Tester': ['Tester logged in.', 'Tester is building an AI.', 'Tester asked to test the system.'], 'entity': 'fact', 'Tester': {'status': 'logged in', 'activity': 'building an AI', 'intent': 'to test the system'}}
✅ Consolidation Passed.

🎉 DIAGNOSTICS COMPLETE. Check 'rua_multi_user_memory.json' in your folder.


In [ ]:
import speech_recognition as sr
import numpy as np
from faster_whisper import WhisperModel
from langchain_ollama import ChatOllama
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.chat_message_histories import ChatMessageHistory
import os
# from memory_management import rua_brain, identity_router # Uncommented this so it works!

def start_rua_hybrid_master():
    # --- Init ---
    stt_model = WhisperModel("small", device="cpu", compute_type="int8")
    local_llm = ChatOllama(model="llama3.2", temperature=0.3)
    cloud_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=GOOGLE_API_KEY)
    session_history = ChatMessageHistory()
    recognizer = sr.Recognizer()

    print("⚡ RUA v7: Multi-User Voice Profiles Enabled.")
    print("👋 RUA: Who am I speaking with today?")

    try:
        with sr.Microphone(sample_rate=16000) as source:
            recognizer.adjust_for_ambient_noise(source)
            while True:
                print(f"\n🟢 Listening ({rua_brain.active_user})...")
                audio = recognizer.listen(source, timeout=None, phrase_time_limit=10)
                
                # --- STT ---
                audio_np = np.frombuffer(audio.get_raw_data(), dtype=np.int16).astype(np.float32) / 32768.0
                segments, _ = stt_model.transcribe(audio_np, beam_size=1)
                full_user_text = "".join([s.text for s in segments]).strip()
                
                if full_user_text:
                    print(f"👤 {rua_brain.active_user}: {full_user_text}")

                    # --- 1. Identity Check ---
                    identity_change = identity_router(full_user_text)
                    if identity_change:
                        print(f"\n{identity_change}")
                    
                    # --- 2. Selective History ---
                    is_follow_up = any(w in full_user_text.lower() for w in ['it', 'they', 'him', 'her', 'why'])
                    
                    # --- 3. Dynamic Multi-User Prompt ---
                    system_prompt = f"""Your name is RUA. 
                    ACTIVE PROFILE: {rua_brain.get_brain_dump()}
                    1. Be witty and personalized to this specific user.
                    2. Under 30 words. No markdown."""

                    messages = [{"role": "system", "content": system_prompt}]
                    if is_follow_up:
                        for msg in session_history.messages[-4:]:
                            messages.append({"role": "user" if msg.type == "human" else "assistant", "content": msg.content})
                    messages.append({"role": "user", "content": full_user_text})

                    # --- 4. Brain Routing ---
                    llm = cloud_llm if "research" in full_user_text.lower() else local_llm
                    print(f"🤖 RUA: ", end="", flush=True)
                    full_response = ""
                    for chunk in llm.stream(messages):
                        content = chunk.content if hasattr(chunk, 'content') else str(chunk)
                        print(content, end="", flush=True)
                        full_response += content

                    session_history.add_user_message(full_user_text)
                    session_history.add_ai_message(full_response)

                    # ---------------------------------------------------------
                    # --- 5. AUTO-CLEANSE TRIGGER (Right here!) ---
                    # ---------------------------------------------------------
                    # Using > 4 for rapid testing. Change back to > 20 later!
                    if len(session_history.messages) > 4:
                        print("\n\n🧠 RUA is consolidating memory...")
                        
                        # Dump the raw chat into episodic memory
                        raw_transcript = " | ".join([f"{msg.type}: {msg.content}" for msg in session_history.messages])
                        rua_brain.memory["users"][rua_brain.active_user]["episodic_memory"].append(raw_transcript)
                        rua_brain.save_to_disk()
                        
                        # Trigger your consolidation logic
                        consolidation_result = rua_brain.consolidate(cloud_llm)
                        print(f"✅ {consolidation_result}")
                        
                        # Wipe the local RAM history
                        session_history.clear() 
                    # ---------------------------------------------------------

                else:
                    print("\r⚪ Silence.")

    except KeyboardInterrupt:
        print(f"\n🛑 Shutting down {rua_brain.active_user}'s session...")
        
        # Save any leftover messages before shutting down
        if len(session_history.messages) > 0:
            raw_transcript = " | ".join([f"{msg.type}: {msg.content}" for msg in session_history.messages])
            rua_brain.memory["users"][rua_brain.active_user]["episodic_memory"].append(raw_transcript)
            rua_brain.save_to_disk()
            
        # Consolidate for the active user
        print(rua_brain.consolidate(cloud_llm))

if __name__ == "__main__":
    start_rua_hybrid_master()

⚡ RUA v7: Multi-User Voice Profiles Enabled.
👋 RUA: Who am I speaking with today?

🟢 Listening (Tester)...
👤 Tester: My name is Yonnel.
👤 [IDENTITY] Switched to profile: Yonnel.

Identity confirmed: Yonnel.
🤖 RUA: Nice to meet you, Yonnel! I'm RUA, your AI buddy - ready to dish out witty banter and helpful info, one conversation at a time!
🟢 Listening (Yonnel.)...
👤 Yonnel.: My favorite animal is Tiger.
🤖 RUA: A roaring good choice, Yonnel! Tigers are majestic, powerful, and oh-so-cool! What's it about them that gets your stripes?
🟢 Listening (Yonnel.)...
👤 Yonnel.: Hello Roa!
🤖 RUA: Yonnel, I think there's been a mix-up - my name is RUA, not Roa! Let's start fresh, what's on your mind?

🧠 RUA is consolidating memory...
✅ Brain optimized for Yonnel..

🟢 Listening (Yonnel.)...
👤 Yonnel.: What is my favorite animal?
🤖 RUA: Yonnel, I'm pretty sure you mentioned the Tiger, but just in case, let me check our previous chat... Ah yes, it's confirmed! You're a tiger fan, no Roa here!
🟢 Listeni